# पाठ ०३ - एजेन्टिक डिजाइन ढाँचाहरू

यस पाठमा, हामी प्रभावकारी AI एजेन्टहरू बनाउनका लागि तीन आधारभूत डिजाइन ढाँचाहरू अन्वेषण गर्छौं:

1. **स्पष्ट एजेन्ट निर्देशनहरू** — एजेन्टको व्यवहारलाई मार्गनिर्देशन गर्ने, भूमिका परिभाषित गर्ने सटीक प्रॉम्प्टहरू तयार पार्नु
2. **पाइडान्टिक मोडेलहरूसँग संरचित आउटपुट** — एजेन्टहरूले पूर्वानुमानित, मान्य डेटा फर्काउने सुनिश्चित गर्नु
3. **एकल जिम्मेवारी एजेन्टहरू** — प्रत्येकले राम्रोसँग एउटा काम गर्ने केन्द्रित एजेन्टहरू डिजाइन गर्नु

हामी प्रत्येक ढाँचालाई **यात्रा गन्तव्य सिफारिस गर्ने** परिदृश्यमा लागू गर्नेछौं, क्रमशः यस्तो प्रणाली निर्माण गर्दै जुन गन्तव्य सुझाव दिन, उपलब्धता जाँच गर्न, र व्यवस्थापन कार्यहरू गर्न सक्दछ।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: स्पष्ट एजेन्ट निर्देशनहरू

सबैभन्दा प्रभावकारी ढाँचा पनि सबैभन्दा सरल हो: तपाईँको एजेन्टका लागि स्पष्ट, विस्तृत निर्देशन लेख्नु।

राम्रो निर्देशनहरूले निर्धारण गर्छन्:
- **को हो** एजेन्ट (व्यक्तित्व र लय)
- **के गर्नु पर्छ** (चरण-द्वारा-चरण जिम्मेवारीहरू)
- **कसरी** व्यवहार गर्नु पर्छ (सीमाहरू र शैली)

तल, हामी एउटा यात्रा कन्सियरज एजेन्ट सिर्जना गर्छौं जसले स्पष्ट निर्देशनहरू पाउँछ जुन उसले उत्पादन गर्ने प्रत्येक प्रतिक्रियालाई आकार दिन्छ।


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: पाइडान्टिक मोडेलहरूसँग संरचित आउटपुट

फ्री-फर्म पाठ वार्तालापको लागि उपयोगी हुन्छ, तर तलका प्रणालीहरूलाई संरचित डाटा आवश्यक हुन्छ।
**पाइडान्टिक मोडेलहरू** लाई **टूल फङ्क्शन** सँग जोड्दै, हामीले गर्न सक्छौं:

- एजेन्टको आउटपुटको लागि निश्चित स्कीमा परिभाषित गर्ने
- प्रतिक्रियाहरू स्वचालित रूपमा प्रमाणित गर्ने
- एजेन्ट परिणामहरूलाई अनुप्रयोग तर्कमा भरपर्दो रूपमा समाहित गर्ने

हामी एउटा टूल पनि परिचय गराउँछौं जसले गन्तव्य विवरणहरू फर्काउँछ ताकि एजेन्टले आफ्नो सिफारिशहरू वास्तविक डाटामा आधारित राख्न सकोस्।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## डिजाइन ३: एकल जिम्मेवारी एजेन्टहरू

जटिल कार्यहरूलाई विभिन्न एकल जिम्मेवारी भएका केन्द्रित एजेन्टहरूमा विभाजन गर्दा फाइदा हुन्छ:

- **गन्तव्य विशेषज्ञ** जसले स्थान र उपलब्धताका बारेमा जानकार हुन्छ
- **लजिस्टिक्स योजनाकार** जसले उडान, होटल, र भ्रमण कार्यक्रम व्यवस्थापन गर्छ

यो सफ्टवेयर इन्जिनियरिङ सिद्धान्त *चिन्ताहरूको पृथकीकरण* संग मिल्दोजुल्दो छ — प्रत्येक एजेन्टलाई स्वतन्त्र रूपमा परीक्षण, मर्मत, र सुधार गर्न सजिलो हुन्छ।


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

यस पाठमा हामीले तीन एजेन्टिक डिजाइन ढाँचाहरू यात्रा सिफारिस गर्ने परिदृश्यमा लागू गर्यौं:

| ढाँचा | मुख्य विचार | लाभ |
|---|---|---|
| **स्पष्ट निर्देशनहरू** | व्यक्तित्व, जिम्मेवारीहरू, र सीमाहरू प्रारम्भमै परिभाषित गर्नुहोस् | सुसंगत, ब्रान्ड अनुसारको एजेन्ट व्यवहार |
| **संरचित आउटपुट** | प्रतिक्रिया ढाँचाको रूपमा Pydantic मोडलहरू प्रयोग गर्नुहोस् | मान्य, मेसिन-द्वारा पढ्न योग्य परिणामहरू |
| **एकल जिम्मेवारी** | प्रत्येक एजेन्टलाई एउटा केन्द्रित काम दिनुहोस् | सजिलै परीक्षण, मर्मत, र संयोजन गर्न सकिन्छ |

यी ढाँचाहरू स्वाभाविक रूपमा संयोजन हुन्छन् — तपाईं स्पष्ट निर्देशनहरूलाई संरचित आउटपुटसँग एकल जिम्मेवारी एजेन्ट भित्र मिश्रण गरेर बलियो, उत्पादन-तयार प्रणालीहरू बनाउन सक्नुहुन्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यस दस्तावेजलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) को प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयास गरिरहेका छौं, तर कृपया जान्नुहोस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धता हुन सक्छ। मौलिक दस्तावेज यसको मूल भाषामा नै अधिकारिक स्रोत मानिनुपर्नेछ। महत्वपूर्ण सूचनाका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट हुने कुनै पनि गलतफहमी वा गलत व्याख्याहरूका लागि हामी जिम्मेवार हुनेछैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
